In [ ]:
# %%
#!/usr/bin/env python3
"""
MultiGrate + Linear Regression TEST Pipeline
Target: Continuous protein levels (e.g., CD45RA)

Assumptions (matches your TRAIN script):
- Model directory: <model_dir>/<split_tag>_multigrate_ols/
    - multivae_model/   (saved by vae.save(...))
    - ols_regressor_<target>.pkl
- Data files under <base_dir>:
    - rna.h5ad, atac.h5ad, adt_minus_<target>.h5ad
- Split indices under <splits_dir>:
    - <split_tag>_test_idx.csv

Outputs saved under:
- <results_dir>/<split_tag>_multigrate_ols/<target_protein>/
    - y_true.npy / y_pred.npy / idx_test.npy
    - test_predictions.csv
    - test_metrics.json
    - (optional) test_embedding.npy
"""

from __future__ import annotations
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import multigrate as mtg
import scvi
import joblib

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score




In [ ]:
# %%
# -----------------------------
# Data Loaders (same style)
# -----------------------------

def load_tea_seq(target_name: str = None, base_dir: str = "../data"):
    base = Path(base_dir)

    print(f"[Loading] RNA and ATAC from {base}")
    rna = ad.read_h5ad(base / "rna.h5ad")
    atac = ad.read_h5ad(base / "atac.h5ad")

    if target_name:
        adt_path = base / f"adt_minus_{target_name}.h5ad"
        print(f"[Loading] ADT-minus for {target_name}: {adt_path.name}")
        adt = ad.read_h5ad(adt_path)
    else:
        print(f"[Loading] Standard ADT from {base / 'adt.h5ad'}")
        adt = ad.read_h5ad(base / "adt.h5ad")

    # sanity: align obs_names
    if not np.array_equal(rna.obs_names.astype(str), atac.obs_names.astype(str)):
        raise ValueError("RNA and ATAC obs_names are not aligned.")
    if not np.array_equal(rna.obs_names.astype(str), adt.obs_names.astype(str)):
        raise ValueError("RNA and ADT obs_names are not aligned.")

    return rna, atac, adt

def load_test_indices(splits_dir: str, split_tag: str, n_cells: int) -> np.ndarray:
    """
    Match the R helper read_index_csv_0based():

    - Read the first numeric column.
    - If it exactly spans 0..n_cells-1 -> treat as 0-based.
    - Else if it exactly spans 1..n_cells -> convert to 0-based.
    - Else default to 0-based (no guessing shift).
    - Validate in-range and no silent clipping.
    """
    path = Path(splits_dir) / f"{split_tag}_test_idx.csv"
    df = pd.read_csv(path, header=0)

    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        raise ValueError(f"No numeric column found in {path}")

    idx = df[num_cols[0]].to_numpy()

    # check NaNs
    if np.isnan(idx).any():
        bad = np.where(np.isnan(idx))[0][:10]
        raise ValueError(f"NaNs in {path} at rows {bad.tolist()}.")

    # require integer-valued
    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        bad = np.where(~np.isclose(idx, idx_int))[0][:10]
        raise ValueError(f"Non-integer indices in {path} at rows {bad.tolist()}: {idx[bad].tolist()}")

    idx = idx_int
    mn, mx = int(idx.min()), int(idx.max())

    # R-style detection
    if mn == 0 and mx == (n_cells - 1):
        idx0 = idx
        base = "0-based (exact sentinel match)"
    elif mn == 1 and mx == n_cells:
        idx0 = idx - 1
        base = "1-based (exact sentinel match) -> converted"
    else:
        idx0 = idx
        base = "0-based (default)"

    # strict range check (no clipping)
    if (idx0 < 0).any() or (idx0 >= n_cells).any():
        bad = np.where((idx0 < 0) | (idx0 >= n_cells))[0][:10]
        raise ValueError(
            f"Out-of-range indices after base handling in {path}. "
            f"min={idx0.min()}, max={idx0.max()}, n_cells={n_cells}. "
            f"Example bad positions {bad.tolist()} with values {idx0[bad].tolist()}."
        )

    print(f"[load_test_indices] {path.name}: {len(idx0)} indices, {base}, range=[{idx0.min()},{idx0.max()}]")
    return idx0.astype(np.int64)


def load_regression_target(target_protein: str, base_dir: str, cell_names: pd.Index):
    path = Path(base_dir) / "response" / f"{target_protein}.csv"
    df_y = pd.read_csv(path, index_col=0)
    return df_y.loc[cell_names].iloc[:, 0].values.astype(np.float32)




In [ ]:
# %%
# -----------------------------
# MultiGrate Setup (must match TRAIN)
# -----------------------------

def build_multigrate_adata(rna, atac, adt):
    """
    Must mirror TRAIN exactly.
    """
    # Ensure float32 for non-RNA (like TRAIN)
    atac.X = atac.layers["norm"].astype(np.float32)
    adt.X  = adt.layers["norm"].astype(np.float32)

    adatas = [[rna], [atac], [adt]]
    adata = mtg.data.organize_multimodal_anndatas(
        adatas=adatas,
        layers=[["norm"], ["norm"], ["norm"]],
    )
    return adata, rna.shape[1]




In [ ]:
# %%
# -----------------------------
# Regression metrics (same style as your scGLUE test)
# -----------------------------

def evaluate_regression(y_true: np.ndarray, y_pred: np.ndarray, with_rank_corr: bool = True) -> dict:
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    eps = 1e-12

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    evs  = float(explained_variance_score(y_true, y_pred))

    yt = (y_true - y_true.mean()) / (y_true.std(ddof=0) + eps)
    yp = (y_pred - y_pred.mean()) / (y_pred.std(ddof=0) + eps)
    pearson_r = float(np.clip((yt * yp).mean(), -1.0, 1.0))

    out = {"rmse": rmse, "mae": mae, "r2": r2, "explained_var": evs, "pearson_r": pearson_r}

    if with_rank_corr:
        try:
            from scipy.stats import spearmanr
            out["spearman_r"] = float(spearmanr(y_true, y_pred).correlation)
        except Exception:
            out["spearman_r"] = np.nan

    return out




In [ ]:
# %%
# -----------------------------
# TEST Function
# -----------------------------

def test_multigrate_regression_ols(
    target_protein: str = "CD45RA",
    split_tag: str = "tea_split3_all_celltypes",
    base_dir: str = "../data",
    splits_dir: str = "../splits",
    dataset_name: str = "tea",
    model_dir: str = "../models",
    results_dir: str = "../results",
    save_embedding: bool = True,
    seed: int = 0,
):
    scvi.settings.seed = seed

    model_root = Path(model_dir).resolve() / f"{split_tag}_multigrate_ols"
    if not model_root.exists():
        raise FileNotFoundError(f"Missing model_root: {model_root}")

    vae_dir = model_root / "multivae_model"
    reg_path = model_root / f"ols_regressor_{target_protein}.pkl"

    if not vae_dir.exists():
        raise FileNotFoundError(f"Missing saved MultiVAE model directory: {vae_dir}")
    if not reg_path.exists():
        raise FileNotFoundError(f"Missing OLS regressor: {reg_path}")

    out_root = Path(results_dir).resolve() / dataset_name / f"{split_tag}_multigrate_ols_{target_protein}"
    out_root.mkdir(parents=True, exist_ok=True)

    # 1) Load data
    print("[Step 1] Loading full data...")
    rna, atac, adt = load_tea_seq(target_name=target_protein, base_dir=base_dir)
    n_cells_total = rna.n_obs

    # 2) Subset TEST
    print("[Step 2] Subsetting TEST split...")
    test_idx = load_test_indices(splits_dir, split_tag, n_cells_total)
    test_names = rna.obs_names[test_idx]

    rna_te  = rna[test_names].copy()
    atac_te = atac[test_names].copy()
    adt_te  = adt[test_names].copy()

    # 3) Build MultiGrate AnnData (must match TRAIN)
    print("[Step 3] Building MultiGrate anndata (TEST)...")
    adata_te, rna_end = build_multigrate_adata(rna_te, atac_te, adt_te)

    # 4) Load VAE + get embeddings
    print("[Step 4] Loading MultiVAE + computing embeddings...")
    mtg.model.MultiVAE.setup_anndata(adata_te, rna_indices_end=rna_end)

    # Load trained model
    vae = mtg.model.MultiVAE.load(str(vae_dir), adata=adata_te)

    # Two safe options:
    # (a) use model API if available; else (b) rely on get_model_output (as in training)
    try:
        Z_test = vae.get_latent_representation()
        # some versions return numpy; some return anndata-like
        Z_test = np.asarray(Z_test)
    except Exception:
        vae.get_model_output()
        if "X_multigrate" not in adata_te.obsm:
            raise RuntimeError("Could not find embeddings in adata_te.obsm['X_multigrate'] after get_model_output().")
        Z_test = np.asarray(adata_te.obsm["X_multigrate"])

    if Z_test.ndim != 2:
        raise RuntimeError(f"Z_test should be 2D; got shape={getattr(Z_test, 'shape', None)}")

    if save_embedding:
        np.save(out_root / "test_embedding.npy", Z_test.astype(np.float32))

    # 5) Load regressor + predict
    print("[Step 5] Predicting with saved OLS regressor...")
    reg = joblib.load(reg_path)

    y_pred = reg.predict(Z_test).astype(np.float32)

    # 6) Load y_true + evaluate
    print("[Step 6] Loading targets + evaluating...")
    y_true = load_regression_target(target_protein, base_dir, test_names)

    report = evaluate_regression(y_true, y_pred, with_rank_corr=True)
    print("[TEST Metrics]", report)

    # 7) Save outputs
    np.save(out_root / "y_true.npy", y_true)
    np.save(out_root / "y_pred.npy", y_pred)
    np.save(out_root / "idx_test.npy", test_idx.astype(int))

    pred_df = pd.DataFrame({"cell": test_names.astype(str), "y_true": y_true, "y_pred": y_pred})
    pred_df.to_csv(out_root / "test_predictions.csv", index=False)

    report_json = dict(report)
    report_json["run"] = {
        "method": "multigrate+ols",
        "task": "regression",
        "target": target_protein,
        "split_tag": split_tag,
        "model_root": str(model_root),
        "vae_dir": str(vae_dir),
        "regressor": str(reg_path),
        "n_test": int(len(test_idx)),
    }
    report_json["timestamp"] = datetime.now().isoformat(timespec="seconds")

    import json
    with open(out_root / "test_metrics.json", "w") as f:
        json.dump(report_json, f, indent=2)

    print("[Saved] outputs under:", out_root)
    print("  - y_true.npy / y_pred.npy / idx_test.npy")
    print("  - test_predictions.csv")
    print("  - test_metrics.json")
    if save_embedding:
        print("  - test_embedding.npy")


# %%
if __name__ == "__main__":
    test_multigrate_regression_ols(target_protein="CD45RA")

# %%
